In [1]:
import torch
from utils import load_splits
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import Dataset

/home/default/projects/fake_news_classification/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#global random seed
SEED = 42
torch.manual_seed(SEED)

In [3]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

True
NVIDIA GeForce RTX 3050 6GB Laptop GPU


device(type='cuda')

In [4]:
X_train, X_val, X_test, y_train, y_val, y_test = load_splits()

1
2
3


In [5]:

def init_model(model_name):
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label={0: "True", 1:"Fake"},
        label2id={"True": 0, "Fake": 0}
    )
    return model


In [11]:
from datasets import Dataset
from transformers import AutoTokenizer

def tokenize_dataset(x, y, model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    dataset = Dataset.from_dict({
        "text": x.fillna("").astype(str).tolist(),
        "labels": y.astype(int).tolist()
    })

    def tokenize(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=256
        )

    return dataset.map(
        tokenize,
        batched=True
    )


In [13]:
ds_train, ds_val = tokenize_dataset(X_train, y_train, "bert-base-uncased"), tokenize_dataset(X_val, y_val, "bert-base-uncased")



Map: 100%|██████████| 9333/9333 [00:00<00:00, 29855.08 examples/s]


In [14]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1,
    }

In [16]:
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

import torch
import gc
import json
from pathlib import Path


def training_loop(learning_rate, model_name, per_device_train_batch_size, gradient_accumulation_step):
    print(f"Training with learning rate: {learning_rate}")

    model = init_model(model_name)

    training_args = TrainingArguments(
        output_dir=f"models/{model_name}_{learning_rate}",
        learning_rate=learning_rate,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=gradient_accumulation_step,
        num_train_epochs=5,
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=5,
        logging_strategy="epoch",
        report_to="none",
        seed=42,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=ds_train,
        eval_dataset=ds_val,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
    )

    trainer.train()

    results = []

    for log in trainer.state.log_history:
        if "eval_f1_macro" in log:
            results.append({
                "learning_rate": learning_rate,
                "epoch": log["epoch"],
                "val_loss": log["eval_loss"],
                "val_accuracy": log["eval_accuracy"],
                "val_precision": log["eval_precision_macro"],
                "val_recall": log["eval_recall_macro"],
                "val_f1": log["eval_f1_macro"],
            })

    # Create logs directory if necessary
    Path("logs").mkdir(parents=True, exist_ok=True)

    # Save a separate JSON file for each learning rate
    log_path = f"logs/{model_name}_lr_{learning_rate}.json"

    with open(log_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4)

    print(f"Saved results to {log_path}")

    del trainer, model
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return results

In [ ]:
ds_train, ds_val = tokenize_dataset(X_train, y_train, "bert-base-uncased"), tokenize_dataset(X_val, y_val, "bert-base-uncased")
all_results = []
for learning_rate in [1e-5, 2e-5, 3e-5]:
    all_results.extend(training_loop(learning_rate, "bert-base-uncased", 2, 8))



Training with learning rate: 1e-05


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1761.00it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,1.802555,0.188255,0.936891,0.935136,0.938333,0.936358
2,1.370885,0.297194,0.941819,0.943423,0.938927,0.940811
3,1.002502,0.325295,0.944927,0.944162,0.944332,0.944246
4,0.692068,0.365882,0.944498,0.943997,0.943560,0.943774
5,0.412077,0.413880,0.943641,0.943211,0.942595,0.942894


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.63s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

Saved results to logs/bert-base-uncased_lr_1e-05.json
Training with learning rate: 2e-05


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1151.98it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

... (3 outputs left out)

Could not reach the Hub ([Errno -3] Temporary failure in name resolution). Using cached commit hash for 'bert-base-uncased'.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1657.96it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro
1,1.732809,0.204087,0.940641,0.938879,0.942746,0.940198
2,1.294750,0.290406,0.945784,0.945843,0.944282,0.945010
3,0.693395,0.395418,0.946105,0.944904,0.946214,0.945512
4,0.279815,0.442291,0.947284,0.946641,0.946598,0.946620
5,0.086357,0.529658,0.946212,0.945289,0.945876,0.945574


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.47s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

Saved results to logs/bert-base-uncased_lr_3e-05.json
